In [1]:
# defining core parameters
playerCount = 4
mainRound = 1
actionRound = 1


In [2]:
class Law:
    def __init__(self, name, ID, effects):
        self.name = name
        self.ID = ID
        self.effects = effects
    def __str__(self):
        return f"{self.name}({self.effects})"
        
A1 = Law("Fiscal Policy 1A", 
         "A1", 
         {"publicSectorSize": 9, 
          "IMFLimit": 2})
B1 = Law("Fiscal Policy 1B", 
         "B1", 
         {"publicSectorSize": 6, 
          "IMFLimit": 2})
C1 = Law("Fiscal Policy 1C", 
         "C1", 
         {"publicSectorSize": 3, 
          "IMFLimit": 1})
A2 = Law("Labour Market 2A", 
         "A2", 
         {"labourPolicy": "L3"})
B2 = Law("Labour Market 2B", 
         "B2", 
         {"labourPolicy": "L3"})
C2 = Law("Labour Market 2C", 
         "C2", 
         {"labourPolicy": "L3"})
A3 = Law("Taxation 3A", 
         "A3",
         {"taxModifier": 3, 
          "taxMultiplier": 2})
B3 = Law("Taxation 3B", 
         "B3",
         {"taxModifier": 2, 
          "taxMultiplier": 1})
C3 = Law("Taxation 3C", 
         "C3",
         {"taxModifier": 1, 
          "taxMultiplier": 0})
A4 = Law("Healthcare & Benefits 4A", 
         "A4", 
         {"stateHealthCost": 0,  
          "taxModifier": 2,  
          "stateSaleLegitimacy": 1,   
          "stateSaleStars": 0})
B4 = Law("Healthcare & Benefits 4B", 
         "B4", 
         {"stateHealthCost": 5,  
          "taxModifier": 1,  
          "stateSaleLegitimacy": 0,   
          "stateSaleStars": 1})
C4 = Law("Healthcare & Benefits 4C", 
         "C4", 
         {"stateHealthCost": 10,  
          "taxModifier": 0,  
          "stateSaleLegitimacy": 0,   
          "stateSaleStars": 0})
A5 = Law("Education 5A", 
         "A5", 
         {"stateEducationCost": 0,  
          "taxModifier": 2,  
          "stateSaleLegitimacy": 1,   
          "stateSaleStars": 1})
B5 = Law("Education 5B", 
         "B5", 
         {"stateEducationCost": 5,  
          "taxModifier": 1,  
          "stateSaleLegitimacy": 0,   
          "stateSaleStars": 1})
C5 = Law("Education 5C", 
         "C5", 
         {"stateEducationCost": 10,  
          "taxModifier": 0,  
          "stateSaleLegitimacy": 0,   
          "stateSaleStars": 0})
A6 = Law("Foreign Trade A6",
        "A6",
        {"tariffLevel": "L1"})
B6 = Law("Foreign Trade B6",
        "B6",
        {"tariffLevel": "L2"})
C6 = Law("Foreign Trade C6",
        "C6",
        {"tariffLevel": "L3"})
A7 = Law("Immigration 7A",
        "A7",
        {"immigrants": 0})
B7 = Law("Immigration 7B",
        "B7",
        {"immigrants": 1})
C7 = Law("Immigration 7C",
        "C7",
        {"immigrants": 2})

activeLaws = {"Fiscal Policy": C1, 
              "Labour Market": B2, 
              "Taxation": A3, 
              "Healthcare & Benefits": B4, 
              "Education": C5, 
              "Foreign Trade": B6, 
              "Immigration": B7}
proposedLaws = {"Fiscal Policy": (None,None),
              "Labour Market": (None,None), 
              "Taxation": (None,None), 
              "Healthcare & Benefits": (None,None), 
              "Education": (None,None), 
              "Foreign Trade": (None,None), 
              "Immigration": (None,None)}




## creating company cards

In [3]:
# defining worker slots

class WorkerSlot:
    def __init__(self, ID, workerClass, skill, occupier = None, parent = None):
        self.ID = ID
        self.workerClass = workerClass
        self.skill = skill
        self.occupier = occupier
        self.parent = parent
        
    def removeOccupier():
        self.occupier = None
        
    def addOccupier(workerID):
        self.occupier = workerID
        

In [4]:
# define company classes
import uuid

class Company:
    # parent company class
    def __init__(self, ID, name, cost, product, baseProduction, salaryOptions = None, salaryLevel = 0, workerSlots = (), strike = False, parentSlot = None):
        self.ID = ID
        self.name = name
        self.cost = cost
        self.product = product
        self.production = baseProduction
        self.workerSlots = workerSlots
        self.salaryOptions = salaryOptions
        self.salaryLevel = salaryLevel
        self.strike = strike
        self.parentSlot = parentSlot
        
    def createWorkerSlot(slotClass, slotSkill):
        existingSlots = list(self.workerSlots)
        existingSlots.append((slotClass, slotSkill))
        self.workerSlots = tuple(existingSlots)
        
    def beginStrike():
        self.strike = True
        
    def endStrike():
        self.strike = False
        
    def setSalaryLevel(salaryLevel):
        self.salaryLevel = salaryLevel
        
    def setParentSlot(parentSlotID):
        self.parentSlot = parentSlotID
        
    
        
class PublicSectorCompany(Company):
    def __init__(self, ID, name, cost, product, baseProduction, salaryOptions = None, salaryLevel = 0, workerSlots = (), strike = False):
        
        #use parent init method
        super().__init__(ID, name, cost, product, baseProduction, salaryOptions, salaryLevel, workerSlots, strike)
        
        #update salary figure and check if operational
        self.salary = salaryOptions["L" + str(salaryLevel)] if salaryOptions else 0
        if not isinstance(workerSlots[0],dict): 
            if workerSlots[0].occupier != None and not strike:
                self.operational = True
            else:
                self.operational = False
        else:
            self.operational = False
        self.owner = "State"
        
    
        
class PrivateSectorCompany(Company):
    def __init__(self, ID, name, owner, cost, product, baseProduction, automated = False, salaryOptions = None, salaryLevel = 0, workerSlots = (), upgradeable = False, upgradeValue = 0, upgraded = False, strike = False):
        # call parent init method
        super().__init__(ID, name, cost, product, baseProduction, salaryOptions, salaryLevel, workerSlots, strike)

        # check if operational
        if not automated:
            if not isinstance(workerSlots[0],dict):
                if workerSlots[0].occupier is not None and not strike:
                    self.operational = True
                else:
                    self.operational = False
            else:
                self.operational = False
        else:
            self.operational = True
                
        # update salary
        self.salary = salaryOptions["L" + str(salaryLevel)] if salaryOptions else 0   

        # automatically update upgrade figure if worker is assigned to middle class company
        if len(workerSlots) > 0:
            if not isinstance(workerSlots[0],dict):
                if workerSlots[-1].occupier is not None:
                    if workerPool[workerSlots[-1].occupier].workerClass == "Working Class" and owner == "Middle Class":
                        upgraded = True

        # update production value - will return 0 if company is disabled
        self.production = (upgraded * upgradeValue + baseProduction) * self.operational

        self.owner = owner
        self.upgradeable = upgradeable
        self.upgradeValue = upgradeValue
        self.upgraded = upgraded 


In [5]:
def defineCompanies():
    
    from copy import deepcopy  
    
    companies = {}

    # define all public companies
    companies["University Hospital"] = PublicSectorCompany(None, "University Hospital", 30, "Health", 6, {"L1": 25, "L2": 30, "L3": 35}, 2, [{"workerClass": "Any", "skill": "Healthcare"}, {"workerClass": "Any", "skill": "Any"},{"workerClass": "Any", "skill": "Any"}])
    companies["Technical University"] = PublicSectorCompany(None, "Technical University", 30, "Education", 6, {"L1": 25, "L2": 30, "L3": 35}, 2, [{"workerClass": "Any", "skill": "Education"}, {"workerClass": "Any", "skill": "Any"},{"workerClass": "Any", "skill": "Any"}])
    companies["National Public Broadcasting"] = PublicSectorCompany(None, "National Public Broadcasting", 30, "Influence", 4, {"L1": 25, "L2": 30, "L3": 35}, 2, [{"workerClass": "Any", "skill": "Media"}, {"workerClass": "Any", "skill": "Any"},{"workerClass": "Any", "skill": "Any"}])
    companies["Regional TV Station"] = PublicSectorCompany(None, "Regional TV Station", 20, "Influence", 3, {"L1": 15, "L2": 20, "L3": 25}, 2, [{"workerClass": "Any", "skill": "Media"}, {"workerClass": "Any", "skill": "Any"},{"workerClass": "Any", "skill": "Any"}])
    companies["Public University"] = PublicSectorCompany(None, "Public University", 20, "Education", 4, {"L1": 15, "L2": 20, "L3": 25}, 2, [{"workerClass": "Any", "skill": "Education"}, {"workerClass": "Any", "skill": "Any"},{"workerClass": "Any", "skill": "Any"}])
    companies["Public Hospital"] = PublicSectorCompany(None, "Public Hospital", 20, "Health", 4, {"L1": 15, "L2": 20, "L3": 25}, 2, [{"workerClass": "Any", "skill": "Healthcare"}, {"workerClass": "Any", "skill": "Any"},{"workerClass": "Any", "skill": "Any"}])

    # define all private companies        
    companies["College"] = PrivateSectorCompany(None, "College", "Capitalists", 16, "Education", 6, False, {"L1": 10, "L2": 20, "L3": 30}, 2, [{"workerClass": "Any", "skill": "Education"},{"workerClass": "Any", "skill": "Any"}], True, 2)
    companies["Clinic"] = PrivateSectorCompany(None, "Clinic", "Capitalists", 16, "Health", 6, False, {"L1": 10, "L2": 20, "L3": 30}, 2, [{"workerClass": "Any", "skill": "Healthcare"},{"workerClass": "Any", "skill": "Any"}], True, 2)
    companies["Shopping Mall"] = PrivateSectorCompany(None, "Shopping Mall", "Capitalists", 16, "Luxuries", 6, False, {"L1": 15, "L2": 20, "L3": 25}, 2, [{"workerClass": "Any", "skill": "Luxury"},{"workerClass": "Any", "skill": "Any"}], True, 2)
    companies["Supermarket"] = PrivateSectorCompany(None, "Supermarket", "Capitalists", 16, "Food", 4, False, {"L1": 15, "L2": 20, "L3": 25}, 2, [{"workerClass": "Any", "skill": "Agriculture"},{"workerClass": "Any", "skill": "Any"}], True, 1)
    companies["Vegetable Farm"] = PrivateSectorCompany(None, "Vegetable Farm", "Capitalists", 15, "Food", 5, False, {"L1": 20, "L2": 25, "L3": 30}, 2, [{"workerClass": "Any", "skill": "Any"},{"workerClass": "Any", "skill": "Any"},{"workerClass": "Any", "skill": "Any"}])
    companies["Hotel"] = PrivateSectorCompany(None, "Hotel", "Capitalists", 15, "Luxuries", 7, False, {"L1": 20, "L2": 25, "L3": 30}, 2, [{"workerClass": "Any", "skill": "Any"},{"workerClass": "Any", "skill": "Any"},{"workerClass": "Any", "skill": "Any"}])
    companies["Electronics Manufacturer"] = PrivateSectorCompany(None, "Electronics Manufacturer", "Capitalists", 25, "Luxuries", 3, True)
    companies["Car Manufacturer"] = PrivateSectorCompany(None, "Car Manufacturer", "Capitalists", 45, "Luxuries", 5, True)
    companies["Medical Village"] = PrivateSectorCompany(None, "Medical Village", "Capitalists", 24, "Health", 9, False, {"L1": 20, "L2": 30, "L3": 40}, 2, [{"workerClass": "Any", "skill": "Healthcare"},{"workerClass": "Any", "skill": "Any"},{"workerClass": "Any", "skill": "Any"}], True, 2)
    companies["Pharmaceutical Company"] = PrivateSectorCompany(None, "Pharmaceutical Company", "Capitalists", 20, "Health", 8, False, {"L1": 20, "L2": 30, "L3": 40}, 2, [{"workerClass": "Any", "skill": "Healthcare"},{"workerClass": "Any", "skill": "Any"},{"workerClass": "Any", "skill": "Any"}], True, 3)
    companies["Automated Grain Farm"] = PrivateSectorCompany(None, "Automated Grain Farm", "Capitalists", 25, "Food", 2, True)
    companies["Fashion Company"] = PrivateSectorCompany(None, "Fashion Company", "Capitalists", 8, "Luxuries", 4, False, {"L1": 10, "L2": 15, "L3": 20}, 2, [{"workerClass": "Any", "skill": "Any"},{"workerClass": "Any", "skill": "Any"}], True, 2)
    companies["Hospital"] = PrivateSectorCompany(None, "Hospital", "Capitalists", 20, "Health", 7, False, {"L1": 10, "L2": 20, "L3": 30}, 2, [{"workerClass": "Any", "skill": "Healthcare"},{"workerClass": "Any", "skill": "Any"}])
    companies["Institute of Technology"] = PrivateSectorCompany(None, "Institute of Technology", "Capitalists", 20, "Education", 8, False, {"L1": 20, "L2": 30, "L3": 40}, 2, [{"workerClass": "Any", "skill": "Education"},{"workerClass": "Any", "skill": "Any"},{"workerClass": "Any", "skill": "Any"}], True, 3)
    companies["TV Station"] = PrivateSectorCompany(None, "TV Station", "Capitalists", 24, "Influence", 4, False, {"L1": 20, "L2": 30, "L3": 40}, 2, [{"workerClass": "Any", "skill": "Media"},{"workerClass": "Any", "skill": "Any"},{"workerClass": "Any", "skill": "Any"}])
    companies["Stadium"] = PrivateSectorCompany(None, "Stadium", "Capitalists", 20, "Luxuries", 8, False, {"L1": 25, "L2": 30, "L3": 35}, 2, [{"workerClass": "Any", "skill": "Luxury"},{"workerClass": "Any", "skill": "Any"},{"workerClass": "Any", "skill": "Any"}], True, 3)
    companies["University"] = PrivateSectorCompany(None, "University", "Capitalists", 24, "Education", 9, False, {"L1": 20, "L2": 30, "L3": 40}, 2, [{"workerClass": "Any", "skill": "Education"},{"workerClass": "Any", "skill": "Any"},{"workerClass": "Any", "skill": "Any"}], True, 2)
    companies["Publishing House"] = PrivateSectorCompany(None, "Publishing House", "Capitalists", 12, "Influence", 3, False, {"L1": 20, "L2": 25, "L3": 30}, 2, [{"workerClass": "Any", "skill": "Any"},{"workerClass": "Any", "skill": "Any"},{"workerClass": "Any", "skill": "Any"}])
    companies["Fast Food Chain"] = PrivateSectorCompany(None, "Fast Food Chain", "Capitalists", 8, "Food", 3, False, {"L1": 10, "L2": 15, "L3": 20}, 2, [{"workerClass": "Any", "skill": "Any"},{"workerClass": "Any", "skill": "Any"}])
    companies["Radio Station"] = PrivateSectorCompany(None, "Radio Station", "Capitalists", 8, "Influence", 2, False, {"L1": 10, "L2": 15, "L3": 20}, 2, [{"workerClass": "Any", "skill": "Any"},{"workerClass": "Any", "skill": "Any"}])
    companies["Automated Dairy Farm"] = PrivateSectorCompany(None, "Automated Dairy Farm", "Capitalists", 45, "Food", 3, True)
    companies["Academy"] = PrivateSectorCompany(None, "Academy", "Capitalists", 20, "Education", 7, False, {"L1": 10, "L2": 20, "L3": 30}, 2, [{"workerClass": "Any", "skill": "Any"},{"workerClass": "Any", "skill": "Any"}])
    companies["Lobbying Firm"] = PrivateSectorCompany(None, "Lobbying Firm", "Capitalists", 16, "Influence", 3, False, {"L1": 10, "L2": 20, "L3": 30}, 2, [{"workerClass": "Any", "skill": "Media"},{"workerClass": "Any", "skill": "Any"}])
    companies["Fish Farm"] = PrivateSectorCompany(None, "Fish Farm", "Capitalists", 20, "Food", 6, False, {"L1": 25, "L2": 30, "L3": 35}, 2, [{"workerClass": "Any", "skill": "Agriculture"},{"workerClass": "Any", "skill": "Any"},{"workerClass": "Any", "skill": "Any"}], True, 1)

    # defining middle class companies
    companies["Regional Radio Station"] = PrivateSectorCompany(None, "Regional Radio Station", "Middle Class", 20, "Influence", 2, False, {"L1": 9, "L2": 12, "L3": 15}, 2, [{"workerClass": "Middle Class", "skill": "Media"},{"workerClass": "Working Class", "skill": "Media"}], True, 2)
    companies["PR Agency"] = PrivateSectorCompany(None, "PR Agency", "Middle Class", 20, "Influence", 3, False, None, 0, [{"workerClass": "Middle Class", "skill": "Media"},{"workerClass": "Middle Class", "skill": "Any"}])
    companies["Local Newspaper"] = PrivateSectorCompany(None, "Local Newspaper", "Middle Class", 14, "Influence", 2, False, {"L1": 6, "L2": 8, "L3": 10}, 2, [{"workerClass": "Middle Class", "skill": "Media"},{"workerClass": "Working Class", "skill": "Any"}], True, 1)
    companies["Electronics Store"] = PrivateSectorCompany(None, "Electronics Store", "Middle Class", 20, "Luxuries", 2, False, {"L1": 9, "L2": 12, "L3": 15}, 2, [{"workerClass": "Middle Class", "skill": "Luxury"},{"workerClass": "Working Class", "skill": "Luxury"}], True, 4)
    companies["Jewelery Store"] = PrivateSectorCompany(None, "Jewelery Store", "Middle Class", 16, "Luxuries", 4, False, None, 0, [{"workerClass": "Middle Class", "skill": "Luxury"},{"workerClass": "Middle Class", "skill": "Any"}])
    companies["Organic Farm"] = PrivateSectorCompany(None, "Organic Farm", "Middle Class", 20, "Food", 2, False, {"L1": 9, "L2": 12, "L3": 15}, 2, [{"workerClass": "Middle Class", "skill": "Agriculture"},{"workerClass": "Working Class", "skill": "Agriculture"}], True, 2)
    companies["Fast Food Restaurant"] = PrivateSectorCompany(None, "Fast Food Restaurant", "Middle Class", 20, "Food", 3, False, None, 0, [{"workerClass": "Middle Class", "skill": "Agriculture"},{"workerClass": "Middle Class", "skill": "Any"}])
    companies["Doctor's Office"] = PrivateSectorCompany(None, "Doctor's Office", "Middle Class", 12, "Health", 2, False, {"L1": 6, "L2": 8, "L3": 10}, 2, [{"workerClass": "Middle Class", "skill": "Healthcare"},{"workerClass": "Working Class", "skill": "Any"}], True, 2)
    companies["Private School"] = PrivateSectorCompany(None, "Private School", "Middle Class", 20, "Education", 2, False, {"L1": 9, "L2": 12, "L3": 15}, 2, [{"workerClass": "Middle Class", "skill": "Education"},{"workerClass": "Working Class", "skill": "Education"}], True, 4)
    companies["Convenience Store"] = PrivateSectorCompany(None, "Convenience Store", "Middle Class", 14, "Food", 2, False, {"L1": 6, "L2": 8, "L3": 10}, 2, [{"workerClass": "Middle Class", "skill": "Agriculture"},{"workerClass": "Working Class", "skill": "Any"}], True, 1)
    companies["Training Centre"] = PrivateSectorCompany(None, "Training Centre", "Middle Class", 16, "Education", 4, False, None, 0, [{"workerClass": "Middle Class", "skill": "Education"},{"workerClass": "Middle Class", "skill": "Any"}])
    companies["Pharmacy"] = PrivateSectorCompany(None, "Pharmacy", "Middle Class", 16, "Health", 4, False, None, 0, [{"workerClass": "Middle Class", "skill": "Healthcare"},{"workerClass": "Middle Class", "skill": "Any"}])
    companies["Games Store"] = PrivateSectorCompany(None, "Game Store", "Middle Class", 12, "Luxuries", 2, False, {"L1": 6, "L2": 8, "L3": 10}, 2, [{"workerClass": "Middle Class", "skill": "Luxury"},{"workerClass": "Working Class", "skill": "Any"}], True, 2)
    companies["Medical Laboratory"] = PrivateSectorCompany(None, "Medical Laboratory", "Middle Class", 20, "Health", 2, False, {"L1": 9, "L2": 12, "L3": 15}, 2, [{"workerClass": "Middle Class", "skill": "Healthcare"},{"workerClass": "Working Class", "skill": "Healthcare"}], True, 4)
    companies["Tutoring Company"] = PrivateSectorCompany(None, "Tutoring Company", "Middle Class", 12, "Education", 2, False, {"L1": 6, "L2": 8, "L3": 10}, 2, [{"workerClass": "Middle Class", "skill": "Education"},{"workerClass": "Working Class", "skill": "Any"}], True, 2)

    # defining working class companies
    companies["Cooperative Farm"] = PrivateSectorCompany(None, "Cooperative Farm", "Working Class", 0, "Food", 2, False, None, 0, [{"workerClass": "Working Class", "skill": "Any"},{"workerClass": "Working Class", "skill": "Any"}, {"workerClass": "Working Class", "skill": "Any"}])

    # updating worker slots to use the class, not dicts
    for companyName, companyObject in companies.items():
        oldWorkerSlots = companyObject.workerSlots
        if oldWorkerSlots is None:
            companyObject.workerSlots = ()
        else:
            newSlots = []
            for slot in oldWorkerSlots:
                newSlots.append(WorkerSlot(None, slot['workerClass'], slot['skill']))
            companyObject.workerSlots = tuple(newSlots)
    
    # compiling company pool for each class
    companyPool = {}

    # capitalist companies
    duplicateCompanies = ["Shopping Mall", "Clinic", "College", "Supermarket"]
    for i in range(len(list(companies.keys()) + duplicateCompanies)):
        if companies[(list(companies.keys()) + duplicateCompanies)[i]].owner == "Capitalists":
            companyObject = deepcopy(companies[(list(companies.keys()) + duplicateCompanies)[i]])
            ID = "capitalistCompany" + str(uuid.uuid4())
            if ID in companyPool.keys():
                print("ID is being overwritten")
            companyPool[ID] = companyObject
            companyObject.ID = ID

    # middle class companies
    duplicateCompanies = ["Convenience Store", "Doctor's Office"]
    for i in range(len(list(companies.keys()) + duplicateCompanies)):
        if companies[(list(companies.keys()) + duplicateCompanies)[i]].owner == "Middle Class":
            companyObject = deepcopy(companies[(list(companies.keys()) + duplicateCompanies)[i]])
            ID = "middleClassCompany" + str(uuid.uuid4())
            if ID in companyPool.keys():
                print("ID is being overwritten")
            companyPool[ID] = companyObject
            companyObject.ID = ID

    # working class companies
    ID = "workingClassCompany" + str(uuid.uuid4())
    companyPool[ID] = deepcopy(companies["Cooperative Farm"])
    companyPool[ID].ID = ID
    ID = "workingClassCompany" + str(uuid.uuid4())
    companyPool[ID] = deepcopy(companies["Cooperative Farm"])
    companyPool[ID].ID = ID
    
    # public sector companies
    duplicateCompanies = ["Public University", "Public University", "Regional TV Station", "Regional TV Station", "Public Hospital", "Public Hospital"]
    for i in range(len(list(companies.keys()) + duplicateCompanies)):
        if companies[(list(companies.keys()) + duplicateCompanies)[i]].owner == "State":
            companyObject = deepcopy(companies[(list(companies.keys()) + duplicateCompanies)[i]])
            ID = "publicSectorCompany" + str(uuid.uuid4())
            if ID in companyPool.keys():
                print("ID is being overwritten")
            companyPool[ID] = companyObject
            companyObject.ID = ID

    
    #updating worker slots with parent companyID
    for key, value in companyPool.items():
        for i in value.workerSlots:
            i.parent = key

    return companyPool

        

### Main ###



# defining functions for moving companies around
companyPool = defineCompanies()

In [23]:
companyPool

{'capitalistCompany3fc7cb42-948f-4cda-bf32-d02dc7f091af': <__main__.PrivateSectorCompany at 0x1f0c84a7650>,
 'capitalistCompanye0a612e7-8eda-4937-b466-6fce463bd01e': <__main__.PrivateSectorCompany at 0x1f0c8464a50>,
 'capitalistCompanyb61a10fc-ce0a-472c-aadc-08311ec664de': <__main__.PrivateSectorCompany at 0x1f0c85a3d10>,
 'capitalistCompanybcadc7ad-a169-4830-9d9e-feda96a71724': <__main__.PrivateSectorCompany at 0x1f0c85a3e50>,
 'capitalistCompanyabae1b63-e8d9-47ea-aa5b-2175bd39d1b6': <__main__.PrivateSectorCompany at 0x1f0c85a3f90>,
 'capitalistCompany8fa39762-2234-4b38-9d9d-459d0515675f': <__main__.PrivateSectorCompany at 0x1f0c85b4150>,
 'capitalistCompanye4061cab-1e39-4baa-b26f-54e3b6ba91e5': <__main__.PrivateSectorCompany at 0x1f0c85b42d0>,
 'capitalistCompany553e037f-badb-4dd6-b19a-c85dd65211cb': <__main__.PrivateSectorCompany at 0x1f0c85b4390>,
 'capitalistCompany47f4f44b-e842-4f1b-9469-c99448957798': <__main__.PrivateSectorCompany at 0x1f0c85b4450>,
 'capitalistCompany2a7bdba3-

In [6]:
# defining trade unions
class TradeUnion:
    def __init__(self, industry):
        self.industry = industry
        self.workerSlots = tuple([WorkerSlot(None, "Working Class", industry, None, industry)])
        

### Main ###



tradeUnions = {"Agriculture": TradeUnion("Agriculture"), 
               "Luxury": TradeUnion("Luxury"),
               "Healthcare": TradeUnion("Healthcare"),
               "Education": TradeUnion("Education"),
               "Media": TradeUnion("Media")}

In [7]:
# building player company decks
def resetCompanyDeck(playerClass):
    
    companyDeck = {}
    if playerClass == "Working Class":
        for key, value in companyPool.items():
            if key[:7] == "working":
                companyDeck[key] = value
        return companyDeck

    elif playerClass == "Middle Class":
        for key, value in companyPool.items():
            if key[:6] == "middle":
                companyDeck[key] = value
        return companyDeck

    elif playerClass == "Capitalists":
        for key, value in companyPool.items():
            if key[:7] == "capital":
                companyDeck[key] = value    
        return companyDeck
    
    elif playerClass == "State":
        for key, value in companyPool.items():
            if key[:6] == "public":
                companyDeck[key] = value    
        return companyDeck
    
    else:
        print("Error building company deck - player not recognised")
        return
    

### Main ###


# setup companies into "cards" and assign them to player decks. setup starting companies
workingClassCompanyDeck = resetCompanyDeck("Working Class")
middleClassCompanyDeck = resetCompanyDeck("Middle Class")
capitalistsCompanyDeck = resetCompanyDeck("Capitalists")
stateCompanyDeck = resetCompanyDeck("State")
   

In [8]:
# defining company slots
def resetCompanySlots():
    
    global companySlots
    
    companySlots = {}
    
    for i in range(1,13,1):
        companySlots["capitalistCompanySlot" + str(i)] = (None, True)

    for i in range(1,9,1):
        companySlots["middleClassCompanySlot" + str(i)] = (None, True)

    for i in range(1,3,1):
        companySlots["workingClassSectorCompanySlot" + str(i)] = (None, True)

    for i in range(1,4,1):
        companySlots["publicSectorCompanySlot" + str(i)] = (None, True)  
    for i in range(4,13,1):
        companySlots["publicSectorCompanySlot" + str(i)] = (None, False)     
    return
    

### Main ###

        
resetCompanySlots()     

## founding and movement of companies

In [9]:
def identifyCompany(companyName, sourcePool):
    # used to find the company ID from its name
    for key, value in sourcePool.items():
        if value.name == companyName:
            return key
    print("Failed to locate company name: " + companyName)   
    return




# def unfoundCompany():

# def buyCompany():
    #found company but with transaction
    
#def sellCompany():
    #unfound with transaction
    
    


In [10]:
def foundCompany(companyID, sourcePool, companySlotID):
    # used to assign a company to a slot and removes it from the pool
    
    global companySlots
    if companySlots[companySlotID][0] is not None:
        print("Failed to found company - slot already occupied")
        print("Company Name: " + sourcePool[companyID].name)
        print("Company Slot: " + companySlotID)
        return
    
    # replace the tuple value in the company slot with the companyID. It retains the open/closed status from the original tuple
    companySlots[companySlotID] = (sourcePool[companyID], companySlots[companySlotID][1])
    sourcePool.pop(companyID)
    return

In [11]:
# put starting companies onto the board
def companySetup(playerCount):
    
    # setup companies into "cards" and assign them to player decks. setup starting companies
    workingClassCompanyDeck = resetCompanyDeck("Working Class")
    middleClassCompanyDeck = resetCompanyDeck("Middle Class")
    capitalistsCompanyDeck = resetCompanyDeck("Capitalists")
    stateCompanyDeck = resetCompanyDeck("State")
    resetCompanySlots()
    
    # get the starting capitalist companies
    toFound = ["College", "Clinic", "Shopping Mall", "Supermarket"]
    for slotKey, slotValue in companySlots.items():
        if slotKey[:10] != "capitalist":
            continue
        for companyKey, companyValue in capitalistsCompanyDeck.items():
            if companyValue.name in toFound:
                foundCompany(companyKey, capitalistsCompanyDeck, slotKey)
                toFound.remove(companyValue.name)
                break
        if len(toFound) == 0:
            break
    
    if playerCount > 2:
        
        # assign middle class companies if neccessary
        toFound = ["Doctor's Office", "Convenience Store"]
        for slotKey, slotValue in companySlots.items():
            if slotKey[:6] != "middle":
                continue
            for companyKey, companyValue in middleClassCompanyDeck.items():
                if companyValue.name in toFound:
                    foundCompany(companyKey, middleClassCompanyDeck, slotKey)
                    toFound.remove(companyValue.name)
                    break
            if len(toFound) == 0:
                break

        # assign > 2 player public sector companies        
        toFound = ["University Hospital", "Technical University", "National Public Broadcasting"]     
        for slotKey, slotValue in companySlots.items():
            if slotKey[:6] != "public" or slotValue[0] is not None or not slotValue[1]:
                continue
            for companyKey, companyValue in stateCompanyDeck.items():
                if companyValue.name in toFound:
                    foundCompany(companyKey, stateCompanyDeck, slotKey)
                    toFound.remove(companyValue.name)
                    break
            if len(toFound) == 0:
                break      
        
    else:        
        # assign 2 player public sector companies
        toFound = ["Public Hospital", "Public University", "Regional TV Station"] 
        for slotKey, slotValue in companySlots.items():
            if slotKey[:6] != "public" or slotValue[0] is not None or not slotValue[1]:
                continue
            for companyKey, companyValue in stateCompanyDeck.items():
                if companyValue.name in toFound:
                    foundCompany(companyKey, stateCompanyDeck, slotKey)
                    toFound.remove(companyValue.name)
                    break
            if len(toFound) == 0:
                break
    
    # assign remaining public sector companies
    toFound = ["Public Hospital", "Public University", "Regional TV Station",
               "Public Hospital", "Public University", "Regional TV Station"]

    for slotKey, slotValue in companySlots.items():
        if slotKey[:6] != "public" or slotValue[0] is not None or slotValue[1]:
            continue
        for companyKey, companyValue in stateCompanyDeck.items():
            if companyValue.name in toFound:              
                foundCompany(companyKey, stateCompanyDeck, slotKey)
                toFound.remove(companyValue.name)
                break
        if len(toFound) == 0:
            break
    
    return


### Main ###


companySetup(playerCount)

## setup workers and worker slots

In [12]:
class Worker:
    def __init__(self, workerClass, skill, assignment = None):
        self.workerClass = workerClass
        self.skill = skill
        self.assignment = assignment

In [13]:
# defining workers
def createWorkerPool(): 
    # this resets all workers, but could create duplicates if not done alongside a reset of all worker slots
    
    workerPool = {}

    i = 1
    for industry in list(tradeUnions.keys()):
        for j in range(5):            
            workerPool.update({"workingClassWorker" + str(i): Worker("Working Class", industry, "off-board")})
            i += 1
    while i < 49:
        workerPool.update({"workingClassWorker" + str(i): Worker("Working Class", "Unskilled", "off-board")})
        i += 1

    i = 1
    for industry in list(tradeUnions.keys()):
        for j in range(5):
            workerPool.update({"middleClassWorker" + str(i): Worker("Middle Class", industry, "off-board")})
            i += 1
    while i < 43:
        workerPool.update({"middleClassWorker" + str(i): Worker("Middle Class", "Unskilled", "off-board")})
        i += 1

    return workerPool


### Main ###


workerPool = createWorkerPool()

In [14]:
def scrapeWorkerSlots(): 
    
    workerSlotsDict = {}
    for key, value in companyPool.items():
        if value.workerSlots is not None:
            newSlots = []
            for slot in value.workerSlots:
                ID = "workerSlot" + str(uuid.uuid4())
                workerSlotsDict[ID] = slot
                newSlots.append(ID)
            value.workerSlots = tuple(newSlots)
    for key, value in tradeUnions.items():
        if value.workerSlots is not None:
            for slot in value.workerSlots:
                ID = "workerSlot" + str(uuid.uuid4())
                workerSlotsDict[ID] = slot
                slot.ID = ID
                newSlots.append(ID)
            value.workerSlots = tuple(newSlots)
                
    return workerSlotsDict


### Main ###


# completing worker setup
import copy
workerSlotsDict = scrapeWorkerSlots()
offBoardWorkers = copy.copy(workerPool)
unemployedWorkers = {}
unemployedWorkerSlots = {}

## worker functions

In [15]:
##OLD CODE
def identifyValidWorker(classReq, skillReq, sourcePool):
    # looks at the target slot and scans a pool of workers for valid candidates. Outputs a list of valid workerIDs.
    # playerClass can be added to filter player's own workers only
    
    validWorkers = []
    
    # compare each worker to the requirements
    for workerID, workerObject in sourcePool.items():
        if classReq != "Any" and skillReq != "Any":
            if workerObject.skill == skillReq and workerObject.workerClass == classReq:
                validWorkers.append(workerID)            
        elif skillReq == "Any" and classReq != "Any":
            if workerObject.workerClass == classReq:
                validWorkers.append(workerID)            
        elif skillReq != "Any" and classReq == "Any":
            if workerObject.skill == skillReq:
                validWorkers.append(workerID)
        else:
            validWorkers.append(workerID)
                
    return validWorkers  

In [16]:
##OLD CODE
def selectWorker(workerClass, skill, validWorkers):
    
    # selects best candidate for a slot
    for workerID in validWorkers:
        if workerPool[workerID].skill == skill and workerPool[workerID].workerClass == workerClass:
            return workerID
    
    for workerID, workerStats in workerPool.items():
        if workerPool[workerID].skill == skill or workerPool[workerID].workerClass == workerClass:
            return workerID
        
    # otherwise return the first worker in the list
    return validWorkers[0]

In [17]:
def selectWorker(classReq, skillReq, sourceDict):
    
    closeMatches = []
    for workerID, workerObject in workerPool.items():
        if workerObject.workerClass == classReq and (workerObject.skill == skillReq or (skillReq == "Any" and workerObject.skill == "Unskilled")) and workerID in sourceDict.keys():
            return workerID
        elif workerObject.workerClass == classReq and workerID in sourceDict.keys() and skillReq == "Any":
            closeMatches.append(workerID)
    if len(closeMatches) == 0:
        print("Error selecting worker - no matches found")
    return closeMatches[0]

        

In [18]:
def assignWorker(workerID, destinationID):
    
    # changes the assignment of the worker to the destination, 
    # removes the worker as an occupier of its previous slot 
    # adds it as an occupier of the new one
    try:
        workerPool[workerID]
    except KeyError:
        print("Failed to locate worker when assigning")    
    else:    
        source = workerPool[workerID].assignment
        workerSlotsDict[source].occupier = None
        workerPool[workerID].assignment = destinationID
        workerSlotsDict[destinationID].occupier = workerID
    
    return 

In [19]:
def assignUnemployedWorker(workerID, destinationID):    
  
    # assigning an unemployed worker from that area will also delete the slot
    
    # check function is valid
    if workerPool[workerID].assignment[:10] != "unemployed": # this refers to the slot name
        print("Error exiting unemployment - worker not assigned unemployed")
        print("Assignment: "+ workerPool[workerID].assignment)
        return

    # move worker
    source = workerPool[workerID].assignment
    assignWorker(workerID, destinationID)

    # delete slot in main log and unemployed area
    workerSlotsDict.pop(source)
    unemployedWorkerSlots.pop(source)

    return

In [20]:
def hireWorker(workerSlotID):
    
    try:
        # find valid worker for a slot and assign them
        workerSlot = workerSlotsDict[workerSlotID]
    except KeyError:
        print("Target workerSlotID not recognised when hiring worker")
    else:
        workerID = selectWorker(workerSlot.workerClass, workerSlot.skill, unemployedWorkers)
        assignUnemployedWorker(workerID, workerSlotID)
    
    return

In [21]:
companySlotName = "capitalistCompanySlot1"
workerSlotName = companySlots[companySlotName][0].workerSlots[0]
print(workerSlotName)

workerSlot130b6394-8a90-4f86-ae88-00b32e301ddb


In [22]:
companySlotName = "capitalistCompanySlot1"
workerSlotName = companySlots[companySlotName][0].workerSlots[0]
workerName = "workingClassWorker16"
oldSlot = "unemployedworkingClassWorker16"

print("Slot occupier: " + str(workerSlotsDict[workerSlotName].occupier))
print("Worker Assignment: " + workerPool[workerName].assignment)
print("Old Slot Occupier: " + str(workerSlotsDict[oldSlot].occupier))

print(workerSlotName)
print(workerSlotsDict[workerSlotName])
hireWorker(workerSlotName)

print("Slot occupier: " + str(workerSlotsDict[companySlots[slotName][0].workerSlots[0]].occupier))
print("Worker Assignment: " + workerPool[workerName].assignment)
print("Old Slot Occupier: " + str(workerSlotsDict[oldSlot].occupier))

Slot occupier: None
Worker Assignment: off-board


KeyError: 'unemployedworkingClassWorker16'

In [ ]:
def checkTradeUnionValidity():

    unionCounter = {}
    for unionName, workerSlots in tradeUnions.items():
        if len(workerSlots) > 0:    
            unionCounter[unionName] = 0

    for companyID, companyObject in companyPool.items():
        if len(companObject.workerSlots) > 0:
            for iSlot in companObject.workerSlots:
                if workerSlotsDict[iSlot].occupier is not None and workerSlotsDict[iSlot].workerClass == "Working Class":
                    unionCounter[workerSlotsDict[iSlot].skill] += 1

    for unionName, workerCount in unionCounter.items():
        if workerCount < 4:
            print("There are less than 4 workers in the " + unionName + " industry. The union worker is now unemployed.")
            fireWorker(tradeUnions[unionName].workerSlots[0].occupier)

    return

In [ ]:
def fireWorker(workerID):
    # move a worker from any slot to the unemployment pool
    
    workerObject = workerPool[workerID]
    
    # create a slot in unemployment pool
    workerSlotsDict["unemployed" + workerID] = WorkerSlot(workerObject.workerClass, workerObject.skill, workerID, "unemployment")
    unemployedWorkerSlots["unemployed" + workerID] = workerSlotsDict["unemployed" + workerID]
    
    # move to new slot
    assignWorker(workerID, "unemployed" + workerID)
    unemployedWorkers[workerID] = workerSlotsDict[workerID]
    
    # check trade union status
    checkTradeUnionValidity()
    
    return

In [ ]:
def birthWorker(workerClass, skill):

    global offBoardWorkers, workerPool
         
    # find eligible worker
    workerID = selectWorker(workerClass, skill, offBoardWorkers)
    print("birthing " + workerID)
    workerObject = workerPool[workerID]
    
    # create a slot in unemployment pool
    workerSlotsDict["unemployed" + workerID] = WorkerSlot(workerObject.workerClass, workerObject.skill, workerID, "unemployment")
    unemployedWorkerSlots["unemployed" + workerID] = workerSlotsDict["unemployed" + workerID]      
    
    # remove worker from off-board pool
    offBoardWorkers.pop(workerID)
    
    # add it to the unemployedWorkers pool
    unemployedWorkers[workerID] = workerPool[workerID]
        
    # assign worker to slot, occupy slot with worker
    workerPool[workerID].assignment = "unemployed" + workerID
    workerSlotsDict["unemployed" + workerID].occupier = workerID

    return

In [ ]:
birthWorker("Working Class", "Education")

In [ ]:
workerSlotsDict

In [ ]:
def setupImmigrationCards():

    import random

    immigrationCards = [
        (("Working Class", "Luxury"), ("Middle Class", "Unskilled")),
        (("Working Class", "Unskilled"), ("Middle Class", "Media")),
        (("Working Class", "Media"), ("Middle Class", "Unskilled")),
        (("Working Class", "Unskilled"), ("Middle Class", "Education")),
        (("Working Class", "Unskilled"), ("Middle Class", "Luxury")),
        (("Working Class", "Media"), ("Middle Class", "Unskilled")),
        (("Working Class", "Agriculture"), ("Middle Class", "Unskilled")),
        (("Working Class", "Agriculture"), ("Middle Class", "Unskilled")),
        (("Working Class", "Unskilled"), ("Middle Class", "Healthcare")),
        (("Working Class", "Unskilled"), ("Middle Class", "Luxury")),
        (("Working Class", "Unskilled"), ("Middle Class", "Education")),
        (("Working Class", "Healthcare"), ("Middle Class", "Education")),
        (("Working Class", "Healthcare"), ("Middle Class", "Unskilled")),
        (("Working Class", "Unskilled"), ("Middle Class", "Healthcare")),
        (("Working Class", "Education"), ("Middle Class", "Unskilled")),
        (("Working Class", "Luxury"), ("Middle Class", "Unskilled")),
        (("Working Class", "Unskilled"), ("Middle Class", "Media")),
        (("Working Class", "Unskilled"), ("Middle Class", "Healthcare")),
        (("Working Class", "Education"), ("Middle Class", "Unskilled")),
        (("Working Class", "Unskilled"), ("Middle Class", "Luxury")),
        (("Working Class", "Unskilled"), ("Middle Class", "Media")),
        (("Working Class", "Unskilled"), ("Middle Class", "Agriculture")),
        (("Working Class", "Unskilled"), ("Middle Class", "Agriculture")),
        (("Working Class", "Unskilled"), ("Middle Class", "Agriculture")),    
    ]

    random.shuffle(immigrationCards)
    
    return immigrationCards

def drawImmigrationCard(player):
    
    global immigrationCards
    
    if player == "Working Class":
        workerStats = immigrationCards[0][0]
    else:
        workerStats = immigrationCards[0][1]
        
    immigrationCards.append(immigrationCards[0])
    immigrationCards.pop(0)
    return workerStats[0], workerStats[1]
    
    
### Main ###

immigrationCards = setupImmigrationCards()

In [ ]:
def setupWorkers(playerCount):
    # initial setup of workers
    
    for i in range(5):
        birthWorker("Working Class", "Unskilled")
    birthWorker("Working Class", "Agriculture")
    birthWorker("Working Class", "Education")
    birthWorker("Working Class", "Healthcare")
    iclass, iskill = drawImmigrationCard("Working Class")
    birthWorker(iclass, iskill)    
    
    if playerCount > 2:
        
        skillDict = {"A": "Agriculture", "E": "Education", "H": "Healthcare", "L": "Luxury", "M": "Media", "U": "Unskilled"}
        
        print("Deploying middle class worker, select skill:\nEducation = E\nHealhcare = H\nLuxury = L\nAgriculture = A\nMedia = M\nUnskilled = U")
        skillLetter = str(input()).upper()
        while skillLetter not in ("A", "E", "H", "L", "M", "U"):
            skillLetter = str(input("Input not understood, try again: "))
        skill = skillDict[skillLetter]
        birthWorker("Middle Class", skill)
        for i in range(4):
            birthWorker("Middle Class", "Unskilled")
        birthWorker("Middle Class", "Education")
        birthWorker("Middle Class", "Luxury")
        birthWorker("Middle Class", "Agriculture")
        birthWorker("Middle Class", "Healthcare")
        iclass, iskill = drawImmigrationCard("Middle Class")
        birthWorker(iclass, iskill)
        
    else:
        birthWorker("Working Class", "Luxury")
        
    if playerCount == 2:
        
        thisClass = "Working Class"
        for companySlotID, CompanySlotObject in companySlots.items():
            if CompanySlotObject[0] is not None:
                if CompanySlotObject[0].name in ["Supermarket", "Shopping Mall", "Public Hospital", "Public University"]:
                    for workerSlotID in CompanySlotObject[0].workerSlots:
                        hireWorker(workerSlotID)

    else:
        thisClass = "Working Class"
        for companySlotID, CompanySlotObject in companySlots.items():
            if CompanySlotObject[0] is not None:
                if CompanySlotObject[0].name in ["Supermarket", "University Hospital", "College"]:
                    for workerSlotID in CompanySlotObject[0].workerSlots:
                        hireWorker(workerSlotID)
        
        thisClass = "Middle Class"
        for companySlotID, CompanySlotObject in companySlots.items():
            if CompanySlotObject[0] is not None:
                if CompanySlotObject[0].name in ["Technical University", "Shopping Mall", "Convenience Store", "Doctor's Office"]:
                    for workerSlotID in CompanySlotObject[0].workerSlots:
                        hireWorker(workerSlotID)    
                        
    return


### Main ###



setupWorkers(playerCount) 
for key, value in unemployedWorkers.items():
    print(value.assignment)

In [ ]:
for companySlotID, CompanySlotObject in companySlots.items():
    if CompanySlotObject[0] is not None:
        if CompanySlotObject[0].name in ["Supermarket", "University Hospital", "College"]:
            print(CompanySlotObject[0].workerSlots)
            for workerSlotID in CompanySlotObject[0].workerSlots:
                print(workerSlotID)
                print(workerSlotsDict[workerSlotID].skill)

In [ ]:
workerSlotsDict["workerSlot56557142-550a-4c6a-b1c7-c7a25c4a448c"].skill

In [ ]:
# defining storage
class Storage:
    def __init__(self, amount, product, limit):
        self.product = product
        self.amount = amount
        self.limit = limit
        
class FreeTradeZone(Storage):
    def __init__(self, amount,  product, limit):
        super().__init__(amount,  product, limit)
        


In [ ]:
# Player setup
privatePriceLevels = {"Food": {"L1": 9, "L2": 12, "L3": 15}, 
               "Health": {"L1": 5, "L2": 8, "L3": 10},
              "Education": {"L1": 5, "L2": 8, "L3": 10},
              "Luxuries": {"L1": 5, "L2": 8, "L3": 10}}
publicPriceLevels = {"Health": activeLaws["Healthcare & Benefits"].effects["stateHealthCost"]
                     , "Education": activeLaws["Education"].effects["stateEducationCost"]
                     , "Influence": 10
                     , "Food": 12
                     , "Luxury": 8
                    }

class Player:
    def __init__(self, name, score, **kwargs):
        self.name = name
        self.score = score
        for key, value in kwargs.items():
            setattr(self, key, value)
        
Capitalists = Player("Capitalists", 0, revenue = 0, capital = 0, trackerPosition = 0,
                     prices = {"Food": "L2", "Luxuries": "L2", "Health": "L2", "Education": "L2"},
                    influence = 0,
                     storages = {"Food": Storage(0, "Food", 8),
                                 "Luxuries" : Storage(0, "Luxuries", 12),
                                 "Health" : Storage(0, "Health", 12),
                                 "Education" : Storage(0, "Education", 12),
                                 "freeTradeFood": FreeTradeZone(0, "Food", 8),
                                 "freeTradeFood": FreeTradeZone(0, "Luxuries", 12)
                                }
                    )

WorkingClass = Player("Working Class", 0, income = 0, prosperity = 0, population = "TBC", ####### <-- UPDATE
                     food = 0, health = 0, education = 0, luxuries = 0, influence = 0
                     )

MiddleClass = Player("Middle Class", 0, income = 0, prosperity = 0, population = "TBC", ####### <-- UPDATE
                     food = 0, health = 0, education = 0, luxuries = 0, influence = 0,
                    prices = {"Food": "L2", "Luxuries": "L2", "Health": "L2", "Education": "L2"},
                    influence = 0,
                     storages = {"Food": Storage(0, "Food", 8),
                                 "Luxuries" : Storage(0, "Luxuries", 12),
                                 "Health" : Storage(0, "Health", 12),
                                 "Education" : Storage(0, "Education", 12)
                                }
                    )

State = Player("State", 0 , treasury = 0, legitimacy = {"Working Class": 1, "Middle Class": 1, "Capitalists": 1},
               legitimacyTokens = {"Working Class": 0, "Middle Class": 0, "Capitalists": 0},
               mediaInfuence = 0, personalInfluence = 0,
               storage = {"Food": Storage(0, "Food", 6),
                          "Luxuries": Storage(0, "Luxuries", 6),
                         "Health": Storage(0, "Health", 6),
                         "Education": Storage(0, "Education", 6)},
               prices = {} # production boosts limits - Add later
              )


In [ ]:
for key, value in companyPool.items():
    if value.name == "Supermarket":
        print(value.workerSlots)

In [ ]:
# deining action cards
class Card:
    __init__(self, player, name, effect, count = 1):
        self.player = player
        self.name = name
        self.effect = effect
        self.count = count
        
cards = {}
cards["Foreign Financial Assistance"] = Card("State", "Foreign Financial Assistance", )